First data glance

In [1]:
# Check if it is running on conda
!which python

/home/codespace/anaconda3/bin/python


In [1]:
import pandas as pd
import numpy as np

# Visualization imports
import seaborn as sns
import matplotlib.pyplot as plt

# Data preparation imports
from sklearn.feature_extraction import DictVectorizer
from sklearn.pipeline import make_pipeline

# Model import
from sklearn.linear_model import LinearRegression
from sklearn.linear_model import Lasso
from sklearn.ensemble import RandomForestRegressor

# Metrics import
from sklearn.metrics import root_mean_squared_error

In [15]:

def prepare_data(
    datapath,
    outcome,
    numerical_features,
    categorical_features,
    pipe=None,
    q_low=None, q_high=None,
    debug_subset=0.0,
):
    # Load data
    df = pd.read_parquet(datapath)
    df.tpep_pickup_datetime = pd.to_datetime(df.tpep_pickup_datetime)
    df.tpep_dropoff_datetime = pd.to_datetime(df.tpep_dropoff_datetime)
    df[outcome] = df.tpep_dropoff_datetime - df.tpep_pickup_datetime
    df[outcome] = df[outcome].dt.total_seconds() / 60
    df[categorical_features] = df[categorical_features].astype(str)

    # Training data preparation
    if pipe is None:
        q_low = df[outcome].quantile(0.02)
        q_high = df[outcome].quantile(0.98)
        df = df[(df[outcome] >= q_low) & (df[outcome] <= q_high)]
        df = df[numerical_features + categorical_features + [outcome]]
        df = df.dropna() # Remove rows with missing values before sampling
        if debug_subset:
            frac = float(debug_subset)
            df = df.sample(frac=frac, random_state=42)
        X = df[numerical_features + categorical_features]
        y = df[outcome]
        dv = DictVectorizer()
        pipe = make_pipeline(dv)
        X_train = pipe.fit_transform(X.to_dict(orient="records"))
    
    # Validation data preparation
    else:
        df = df[numerical_features + categorical_features + [outcome]]
        df = df.dropna() # Remove rows with missing values before sampling
        if debug_subset:
            frac = float(debug_subset)
            df = df.sample(frac=frac, random_state=42)
        df = df[(df[outcome] >= q_low) & (df[outcome] <= q_high)]
        X = df[numerical_features + categorical_features]
        y = df[outcome]
        X_train = pipe.transform(X.to_dict(orient="records"))

    return X_train, y, pipe, q_low, q_high

def train_model(X_train, y, y_test, model):
    model.fit(X_train, y)
    y_pred = model.predict(X_train)
    rmse = root_mean_squared_error(y, y_pred)
    print(f"RMSE: {rmse:.2f}")

In [16]:
NUMERICAL_FEATURES = ["trip_distance", "passenger_count", "tip_amount", "total_amount", "tolls_amount"]
CATEGORICAL_FEATURES = ['PULocationID', 'DOLocationID', "payment_type"]
OUTCOME = 'duration'
PIPE = None

train_path = '/workspaces/MLOps-zoomcamp/data/yellow_tripdata_2020-01.parquet'
test_path = '/workspaces/MLOps-zoomcamp/data/yellow_tripdata_2020-02.parquet'

X_train, y_train, pipe, q_low, q_high = prepare_data(train_path, OUTCOME, NUMERICAL_FEATURES, CATEGORICAL_FEATURES, pipe=None, debug_subset=0.5)
X_test, y_test, _, _, _ = prepare_data(test_path, OUTCOME, NUMERICAL_FEATURES, CATEGORICAL_FEATURES, pipe, q_low, q_high, debug_subset=0.5)

In [17]:
# Get dimensions of train and val data
X_test

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 24019134 stored elements and shape (3002393, 523)>

In [18]:
model = LinearRegression()
#model = RandomForestRegressor(n_estimators=100, random_state=42)
#model = Lasso(alpha = 0.01)
train_model(X_train, y_train, y_test, model)

RMSE: 4.04


In [19]:
np.std(y_test)

8.176075374680078